# 3b — RESTORE efficacy diagnostic (mxnorm)

RESTORE ships no before/after efficacy measure. This stage runs the **`mxnorm`** R package
(`Phenocycler_Analysis/scripts/R/restore_mxnorm_diagnostics.R`, the pipeline's `restore_diag` stage) to
quantify — per `[target ← reference]` pair — whether RESTORE actually harmonized the cohort, comparing the
REDSEA-corrected "before" (`cells_redsea`) with the RESTORE-normalized "after":

- **Anderson–Darling** distributional statistic across donors (`kSamples` on per-slide `bkde` curves),
- **Otsu discordance** (per-slide vs global threshold — cross-donor positivity consistency),
- **slide-level variance** decomposition (`lme4`, de-confounded *within* disease stage).

**Read these as descriptors + over-correction alarms, not a "lower = better" gate.** One PhenoCycler image
per donor means `slide = donor = image`, so the slide axis conflates donor *biology* (ND→Aab+→T1D insulin
loss) with technical drift — a large AD/variance drop on INS/GCG/SST is **suspicious**, not a win. The real
per-pair **acceptance** signal is **per-donor**: does RESTORE's applied cutoff track *each donor's own* raw
Otsu valley (`fig_perdonor_residual.png` / `restore_perdonor_threshold_residual.csv`)? Systematic overshoot
there is the curation signal that feeds the threshold-statistic choice, and the insulin-loss **guardrail**
(`restore_biology_guardrail.csv`) confirms the biology survives normalization.

> **Gated:** meaningful cohort-wide only **after the faithful 22-donor RESTORE re-run** —
> `positive_fractions.csv` is 6380-only until then, and cross-slide metrics need ≥2 donors. The compute
> cell prints a short "gated" note until that lands. Runs under **R 4.5.1** (`/usr/local/bin/Rscript`;
> install deps once with `/usr/local/bin/Rscript Phenocycler_Analysis/scripts/R/install_mxnorm.R`).

In [1]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)
# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few


cfg = PipelineConfig(
    data_dir       = REPO / "data",
    donor_metadata = pathlib.Path("/home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx"),
    # --- RESTORE efficacy diagnostic (this step; mxnorm, runs under R 4.5.1) ---
    restore_diag_subsample = 20000,  # cells/donor for the mxnorm diagnostic (stratified by cell_region)
    restore_diag_seed      = 0,
)
print("data_dir          :", cfg.data_dir)
print("donor_metadata    :", cfg.donor_metadata)
print("restore_mxnorm_dir:", cfg.restore_mxnorm_dir)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))


data_dir          : /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data
donor_metadata    : /home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx
restore_mxnorm_dir: /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/mxnorm
donors: 22 (all) -> ['115', '6374', '6380', '6436', '6442', '6450'] ...


In [2]:
# NOTE: restore_diag runs on every donor in positive_fractions.csv; DONORS is not wired to the R diagnostic.
# RESTORE before/after efficacy via mxnorm (scripts/R/restore_mxnorm_diagnostics.R, run under R 4.5.1).
# GATED on the 22-donor re-run: cross-slide metrics need >=2 donors in positive_fractions.csv, so this
# prints a short "gated" note until that lands. Idempotent (force=False skips if the efficacy CSV exists).
from phenocycler.pipeline import run_pipeline
try:
    run_pipeline(cfg, only=['restore_diag'], force=False)
except Exception as e:
    pf = cfg.restore_dir / 'positive_fractions.csv'
    print(f"[gated] {e}\n        (expected until the 22-donor RESTORE re-run fills {pf};\n"
          "         cross-slide metrics need >=2 donors.)")


=== restore_diag ===
[restore_diag] /usr/local/bin/Rscript /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/Phenocycler_Analysis/scripts/R/restore_mxnorm_diagnostics.R --threshold-csv /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/positive_fractions.csv --cells-redsea-dir /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/cells_redsea --cells-dir /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/cells --donor-metadata /home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx --outdir /home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/mxnorm --subsample 20000 --seed 0
[cfg] threshold_csv=/home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/positive_fractions.csv 
[cfg] cells_redsea=/home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/cells_redsea  cells=/home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/cells 
[cfg] outdir=/home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/mxnorm  subsample=20000  seed=0  um

Error in file(file, "rt") : cannot open the connection
Calls: read.csv -> read.table -> file
In addition: Warning message:
In file(file, "rt") :
  cannot open file '/home/smith6jt/IO60panc2nd/Islet-Explorer-Senior/data/restore_redsea/positive_fractions.csv': No such file or directory
Execution halted


In [3]:
from IPython.display import Image, display
import pandas as pd

MXN = cfg.restore_mxnorm_dir
eff = MXN / "restore_mxnorm_efficacy.csv"
if eff.exists():
    e = pd.read_csv(eff)
    cols = [c for c in ["marker", "median_abs_residual", "frac_overshoot",
                        "disc_delta", "ad_delta", "var_delta"] if c in e.columns]
    print("RESTORE efficacy per pair (largest per-donor threshold-vs-valley residual first):")
    display(e[cols].round(3))
    for f in ("fig_perdonor_residual.png", "fig_pair_deltas.png",
              "fig_umap_by_status.png", "fig_discordance.png"):
        if (MXN / f).exists():
            display(Image(filename=str(MXN / f)))
else:
    print("(no mxnorm efficacy output yet — meaningful after the 22-donor RESTORE re-run)")

(no mxnorm efficacy output yet — meaningful after the 22-donor RESTORE re-run)
